In [ ]:
import json
import pandas as pd
import statsmodels.api as sm

# sub_category normalize
def normalize_list(v):
    if isinstance(v, list):
        return [str(x) for x in v]
    return [str(v)]

def extract_minimal(path):
    data = json.load(open(path, "r"))
    model = data["model_name"]
    
    rows = []
    for it in data["results"]:
        rows.append({
            "model_name": str(model),
            "category": str(it.get("category")),
            "sub_category": normalize_list(it.get("sub_category")),
            "full_entropy": float(it.get("full_entropy")),
            "choice_entropy": float(it.get("choice_entropy")),
            "is_correct": int(it.get("is_correct"))
        })
    return pd.DataFrame(rows)


df = pd.concat([
    extract_minimal("exaone_result.json"),
    extract_minimal("kanana_result.json"),
    extract_minimal("midm_result.json"),
    extract_minimal("tri_result.json")
], ignore_index=True)


unique_subcats = sorted({x for lst in df["sub_category"] for x in lst})
print("unique sub_category:", unique_subcats)


for cat in unique_subcats:
    df[f"subcat_{cat}"] = df["sub_category"].apply(lambda lst: 1 if cat in lst else 0)

df = df.drop(columns=["sub_category"])

df = pd.get_dummies(df, columns=["model_name", "category"], drop_first=True)

y = df["is_correct"]
X = df.drop(columns=["is_correct"])

X = sm.add_constant(X)
model = sm.OLS(y.astype(float), X.astype(float)).fit()

print(model.summary())


unique sub_category: ['age', 'appearance', 'disability', 'gender', 'nationality', 'orientation', 'race', 'religion']
                            OLS Regression Results                            
Dep. Variable:             is_correct   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     1.465
Date:                Sat, 29 Nov 2025   Prob (F-statistic):              0.144
Time:                        21:38:33   Log-Likelihood:                -10.396
No. Observations:                 160   AIC:                             46.79
Df Residuals:                     147   BIC:                             86.77
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025    

In [2]:

df.to_csv("full_v1.csv", index= False )

In [5]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_df = pd.DataFrame({
    "feature": X.columns,
    "shap_value": mean_abs_shap
}).sort_values("shap_value", ascending=False)

fig = go.Figure(go.Bar(
    x=shap_df["shap_value"],
    y=shap_df["feature"],
    orientation='h'
))

fig.update_layout(
    title="SHAP Feature Importance",
    xaxis_title="Mean |SHAP value|",
    yaxis_title="Feature",
    template="plotly_white",
    height=800
)

fig.show()


In [10]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# SHAP 값 DataFrame
values = shap_values.values
features = X.columns
df_shap = pd.DataFrame(values, columns=features)

# 절대값 기준 중요도 정렬
ordered_cols = df_shap.abs().mean().sort_values(ascending=False).index
df_shap = df_shap[ordered_cols]

# Plotly Horizontal Violins
fig = go.Figure()

for col in df_shap.columns:
    fig.add_trace(go.Violin(
        y=[col]*len(df_shap),      # feature 이름을 y축으로 반복
        x=df_shap[col],            # SHAP 값을 x축
        name=col,
        box_visible=True,
        meanline_visible=True,
        orientation='h',           # 🎯 핵심: 가로 방향 바이올린
        spanmode="hard"
    ))

fig.update_layout(
    title="SHAP Value Violin Plot (Horizontal)",
    xaxis_title="SHAP Value",
    yaxis_title="Feature",
    template="plotly_white",
    width=1400,
    height=900,
)

fig.show()


- 이 SHAP violin plot은 모델 종류(model_name)와 특정 편향 카테고리(sub_category: religion, race 등)가 정답 예측에 가장 핵심적인 기여를 하며, 
entropy 기반 feature는 correctness 예측에 의미 있는 영향을 주지 않음을 보여준다.